## Лабораторная работа № 2

## Машинное обучение на больших данных с использованием фреймворка Apache Spark и библиотеки SparkML

### Часть 2

В данной части работы рассмотрены:
* подготовка признаков для решения задачи **бинарной классификации**;
* создание и обучение модели **логистической регрессии**;
* подбор гиперпараметров с помощью кросс-валидации;
* оценка качества модели на тестовой выборке.

Датасет Steam Games из лабораторной № 1. Источник данных — подготовленный Parquet `steam_games_cleaned.parquet`. Целевой признак формируется на основе пользовательских отзывов: игры с долей положительных отзывов не менее 70% относятся к положительному классу. Признаки, из которых непосредственно формируется целевая переменная, не используются при обучении, чтобы избежать утечки целевой информации.

#### Запуск `Spark`-сессии

Подключаем необходимые библиотеки.

In [1]:
from pyspark import SparkConf
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, Imputer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.tuning import CrossValidator, CrossValidatorModel, ParamGridBuilder

Сформируем объект конфигурации для `Apache Spark`, указав параметры локального запуска. При работе в учебном кластере параметры конфигурации можно заменить на используемые в нём.

In [2]:
def create_spark_configuration() -> SparkConf:
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """
    conf = SparkConf()
    conf.setAppName("Lab 2 Steam Logistic Regression")
    conf.setMaster("local[2]")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.sql.shuffle.partitions", "8")
    return conf

Создаём сам объект конфигурации.

In [3]:
conf = create_spark_configuration()

Создаём и выводим на экран сессию `Apache Spark`.

In [4]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark

#### Загрузка датасета

Укажем путь к базе данных в формате Parquet, подготовленной в первой лабораторной работе.

In [5]:
PARQUET_PATH = "steam_games_cleaned.parquet" 

Прочитаем подготовленный датасет и загрузим его в `Spark DataFrame`.

In [6]:
df = spark.read.parquet(PARQUET_PATH)

Выведем несколько строк датасета на экран.

In [7]:
df.show(5, truncate=60)

+-------+----------------------+------------+------------------+----------------+--------+--------+---------------+------------------------+---------------------+-------------------+------------+----------+--------------+-------------------------------------------+---------------------------+---------------------------+------------------------------------------------------------+
| app_id|                  name|release_date|             price|estimated_owners|positive|negative|recommendations|average_playtime_forever|steam_store_available|steam_spy_available|release_year|num_genres|num_categories|                                     genres|                 developers|                 publishers|                                                  categories|
+-------+----------------------+------------+------------------+----------------+--------+--------+---------------+------------------------+---------------------+-------------------+------------+----------+--------------+-------------

Вспомним описание основных столбцов и параметры датасета, проанализированные в первой лабораторной работе.

| Название столбца | Расшифровка | Тип признака |
| ---------------- | ----------- | ------------ |
| `app_id` | Идентификатор игры в Steam | идентификатор, исключён |
| `name` | Название игры | текстовый, исключён |
| `release_date` | Дата выхода игры | текстовый, исключён |
| `price` | Цена игры | количественный |
| `estimated_owners` | Оценка количества владельцев | текстовый диапазон, исключён |
| `positive` | Количество положительных отзывов | используется для создания целевой переменной, исключён из признаков |
| `negative` | Количество отрицательных отзывов | используется для создания целевой переменной, исключён из признаков |
| `recommendations` | Количество рекомендаций | количественный |
| `average_playtime_forever` | Среднее время игры за всё время | количественный |
| `steam_store_available` | Наличие игры в магазине Steam | бинарный |
| `steam_spy_available` | Наличие данных Steam Spy | бинарный |
| `release_year` | Год выпуска игры | количественный |
| `num_genres` | Количество жанров | количественный |
| `num_categories` | Количество категорий | количественный |
| `genres`, `developers`, `publishers`, `categories` | Списки жанров, разработчиков, издателей и категорий | массивы строк, исключены |

Вспомним схему данных.

In [8]:
df.printSchema()

root
 |-- app_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- estimated_owners: string (nullable = true)
 |-- positive: double (nullable = true)
 |-- negative: double (nullable = true)
 |-- recommendations: double (nullable = true)
 |-- average_playtime_forever: double (nullable = true)
 |-- steam_store_available: boolean (nullable = true)
 |-- steam_spy_available: boolean (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- num_genres: double (nullable = true)
 |-- num_categories: double (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- developers: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- publishers: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)



Вычислим количество строк в датафрейме.

In [9]:
print("Количество строк:", df.count())

Количество строк: 129587


#### Постановка задачи бинарной классификации

Для датасета Steam Games требуется построить модель **логистической регрессии** (`LogisticRegression`), которая классифицирует игры на две группы:
* **положительно оценённые** — доля положительных отзывов не менее 70%;
* **остальные игры** — доля положительных отзывов менее 70%.

Доля положительных отзывов вычисляется по формуле:

\[
positive\_ratio = \frac{positive}{positive + negative}
\]

Порог 70% выбран для формирования бинарной целевой переменной и не является исходной меткой датасета. Записи без отзывов исключаются, поскольку для них невозможно определить пользовательский рейтинг.

Для оценки качества модели используем ROC-AUC, accuracy, precision, recall и F1.

#### Создание целевого признака

Проверим наличие столбцов с количеством положительных и отрицательных отзывов. Они необходимы для вычисления целевой переменной.

In [10]:
required_columns = ["positive", "negative"]
missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"В Parquet отсутствуют столбцы {missing_columns}. "
        "Необходимо использовать результат очистки ЛР1."
    )

Приведём количество отзывов к числовому типу. Исключим строки с отсутствующими значениями или с нулевым общим количеством отзывов.

In [11]:
df = (
    df.withColumn("positive", F.col("positive").cast("double"))
      .withColumn("negative", F.col("negative").cast("double"))
)

df = df.filter(
    F.col("positive").isNotNull()
    & F.col("negative").isNotNull()
    & (~F.isnan(F.col("positive")))
    & (~F.isnan(F.col("negative")))
    & ((F.col("positive") + F.col("negative")) > 0)
)

Рассчитаем долю положительных отзывов и сформируем целевой столбец `label`. Значение `1.0` соответствует положительно оценённой игре, `0.0` — остальным играм.

In [12]:
POSITIVE_THRESHOLD = 0.70

df = df.withColumn(
    "positive_ratio",
    F.col("positive") / (F.col("positive") + F.col("negative"))
)

df = df.withColumn(
    "label",
    (F.col("positive_ratio") >= POSITIVE_THRESHOLD).cast("double")
)

print("Размер выборки после формирования целевого признака:", df.count())
df.groupBy("label").count().orderBy("label").show()

Размер выборки после формирования целевого признака: 81691
+-----+-----+
|label|count|
+-----+-----+
|  0.0|25599|
|  1.0|56092|
+-----+-----+



#### Подготовка и кодирование признаков

Для обучения модели используем числовые характеристики игры, которые не участвуют в формировании целевой переменной. Столбцы `positive`, `negative` и `positive_ratio` исключаем из признаков, чтобы избежать утечки целевой информации.

Идентификаторы, название игры, текстовые списки и массивы также не передаются модели.

In [13]:
candidate_features = [
    "price",
    "release_year",
    "num_genres",
    "num_categories",
    "recommendations",
    "average_playtime_forever"
]

numeric_features = [
    column for column in candidate_features
    if column in df.columns
]

if not numeric_features:
    raise ValueError(
        "Не найдены подходящие числовые признаки в Parquet."
    )

print("Признаки модели:", numeric_features)

Признаки модели: ['price', 'release_year', 'num_genres', 'num_categories', 'recommendations', 'average_playtime_forever']


Приведём используемые признаки к числовому типу. Значения `NaN` преобразуем в пропуски, которые позднее будут заполнены медианными значениями с помощью трансформера `Imputer`.

In [14]:
for column in numeric_features:
    df = df.withColumn(column, F.col(column).cast("double"))
    df = df.withColumn(
        column,
        F.when(F.isnan(F.col(column)), None).otherwise(F.col(column))
    )

df = df.filter(F.col("label").isNotNull())

df.select(numeric_features + ["label"]).describe().show()

+-------+-----------------+------------------+------------------+-----------------+------------------+------------------------+------------------+
|summary|            price|      release_year|        num_genres|   num_categories|   recommendations|average_playtime_forever|             label|
+-------+-----------------+------------------+------------------+-----------------+------------------+------------------------+------------------+
|  count|            81691|             81691|             81691|            81691|             81691|                   81691|             81691|
|   mean|8.397091680064909|2020.5095420548164|3.6419923859421477|6.269429924961134|1622.1766530945854|      114.39203829063177|0.6866362267569255|
| stddev|12.15800216676097|3.3074943778450816|2.1548048872993344|5.463308484150718| 27910.23789200986|       2414.460042455915|0.4638637222356876|
|    min|              0.0|            1997.0|               1.0|              1.0|               0.0|                

Разделим датасет на обучающую и тестовую выборки в соотношении 80 к 20. Фиксируем `seed`, чтобы разбиение можно было воспроизвести.

In [15]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=11)

train_df = train_df.cache()
test_df = test_df.cache()

print(f"Train dataset size: {train_df.count()}")
print(f"Test dataset size: {test_df.count()}")

print("Распределение классов в обучающей выборке:")
train_df.groupBy("label").count().orderBy("label").show()

print("Распределение классов в тестовой выборке:")
test_df.groupBy("label").count().orderBy("label").show()

Train dataset size: 65133
Test dataset size: 16558
Распределение классов в обучающей выборке:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|20463|
|  1.0|44670|
+-----+-----+

Распределение классов в тестовой выборке:
+-----+-----+
|label|count|
+-----+-----+
|  0.0| 5136|
|  1.0|11422|
+-----+-----+



#### Создание ML-конвейера

Создадим конвейер обработки данных, включающий заполнение пропусков, объединение признаков и модель логистической регрессии.

Для заполнения пропусков используем `Imputer` со стратегией `median`. Затем `VectorAssembler` объединит подготовленные признаки в единый вектор `features`, который будет передан модели `LogisticRegression`.

In [16]:
imputed_features = [
    f"{feature}_imputed" for feature in numeric_features
]

imputer = Imputer(
    inputCols=numeric_features,
    outputCols=imputed_features,
    strategy="median"
)

assembler = VectorAssembler(
    inputCols=imputed_features,
    outputCol="features",
    handleInvalid="skip"
)

logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100
)

pipeline = Pipeline(stages=[
    imputer,
    assembler,
    logistic_regression
])

Выведем описание созданного конвейера.

In [17]:
pipeline

Pipeline_c90e7c7c6964

#### Обучение модели

Выполним **подбор гиперпараметров** модели логистической регрессии с помощью кросс-валидации на сетке.

В качестве настраиваемых гиперпараметров рассмотрим коэффициент регуляризации (`regParam`) и параметр смешивания L1- и L2-регуляризации (`elasticNetParam`).

Создадим сетку параметров для кросс-валидации, используя объект `LogisticRegression`.

In [18]:
param_grid = (
    ParamGridBuilder()
    .addGrid(logistic_regression.regParam, [0.01, 0.1, 1.0])
    .addGrid(logistic_regression.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

Создадим экземпляр `BinaryClassificationEvaluator` для оценки качества моделей по метрике `areaUnderROC` (ROC-AUC). Эта метрика оценивает способность модели разделять два класса при различных порогах классификации.

In [19]:
cv_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

Создадим объект `CrossValidator`. Для кросс-валидации используем 3 фолда.

In [20]:
cross_validator = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,
    numFolds=3,
    parallelism=2,
    seed=11
)

Обучаем конвейер с использованием кросс-валидации. В результате будет выбрана модель с наибольшим значением ROC-AUC на проверочных фолдах.

In [21]:
cv_model = cross_validator.fit(train_df)

Выведем параметры **лучшей** модели, определённой в ходе кросс-валидации.

In [22]:
def get_best_model_params(cv_model: CrossValidatorModel) -> dict:
    """
    Получает параметры лучшей модели LogisticRegression.

    Args:
        cv_model (CrossValidatorModel): Обученная модель кросс-валидации.

    Returns:
        dict: Значения основных гиперпараметров лучшей модели.
    """
    best_model = cv_model.bestModel
    logistic_model = best_model.stages[-1]

    return {
        "regParam": logistic_model.getRegParam(),
        "elasticNetParam": logistic_model.getElasticNetParam(),
        "maxIter": logistic_model.getMaxIter()
    }

In [23]:
for key, value in get_best_model_params(cv_model=cv_model).items():
    print(f"{key}: {value}")

regParam: 0.01
elasticNetParam: 0.0
maxIter: 100


#### Проверка обобщающей способности модели

Выполним предсказания на тестовой выборке. Для каждой игры модель рассчитает вероятность принадлежности к классам и итоговую предсказанную метку.

In [24]:
predictions = cv_model.transform(test_df).cache()

predictions.select(
    "label", "probability", "prediction"
).show(10, truncate=False)

+-----+----------------------------------------+----------+
|label|probability                             |prediction|
+-----+----------------------------------------+----------+
|0.0  |[0.3114339553844376,0.6885660446155624] |1.0       |
|0.0  |[0.33611475311233946,0.6638852468876606]|1.0       |
|1.0  |[0.340495653873738,0.659504346126262]   |1.0       |
|0.0  |[0.39112974296335057,0.6088702570366494]|1.0       |
|1.0  |[0.2900804033787754,0.7099195966212246] |1.0       |
|1.0  |[0.34802235425288286,0.6519776457471171]|1.0       |
|0.0  |[0.17955907340531396,0.820440926594686] |1.0       |
|0.0  |[0.3254153164611805,0.6745846835388195] |1.0       |
|0.0  |[0.3925466583554358,0.6074533416445642] |1.0       |
|1.0  |[0.31854973272484116,0.6814502672751588]|1.0       |
+-----+----------------------------------------+----------+
only showing top 10 rows



Создадим функцию оценки модели: расчёта значения выбранной метрики для датафрейма с предсказаниями.

In [25]:
def evaluate_classification_model(
    data: DataFrame,
    metric_name: str
) -> float:
    """
    Оценивает модель классификации с использованием указанной метрики.

    Args:
        data (DataFrame): DataFrame с предсказаниями и фактическими метками.
        metric_name (str): Название метрики.

    Returns:
        float: Значение выбранной метрики.
    """
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric_name
    )
    return evaluator.evaluate(data)

Оценим модель на тестовой выборке по метрике `ROC-AUC`.

In [26]:
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

test_auc = auc_evaluator.evaluate(predictions)
print(f"ROC-AUC on test data: {test_auc:.4f}")

ROC-AUC on test data: 0.6203


Рассчитаем дополнительные метрики классификации: `accuracy`, `weightedPrecision`, `weightedRecall` и `f1`.

In [27]:
classification_metrics = [
    "accuracy",
    "weightedPrecision",
    "weightedRecall",
    "f1"
]

for metric_name in classification_metrics:
    metric_value = evaluate_classification_model(
        predictions,
        metric_name
    )
    print(f"{metric_name}: {metric_value:.4f}")

accuracy: 0.6893
weightedPrecision: 0.6344
weightedRecall: 0.6893
f1: 0.5958


Посмотрим на распределение фактических и предсказанных классов. Таблица позволяет оценить, какие классы модель определяет правильно и где возникают ошибки классификации.

In [28]:
predictions.groupBy(
    "label", "prediction"
).count().orderBy(
    "label", "prediction"
).show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|  338|
|  0.0|       1.0| 4798|
|  1.0|       0.0|  347|
|  1.0|       1.0|11075|
+-----+----------+-----+



#### Сохранение модели

Сохраним лучшую обученную модель конвейера в локальную директорию. При необходимости путь можно изменить.

In [29]:
MODEL_PATH = "steam_logistic_classification_model"

cv_model.bestModel.write().overwrite().save(MODEL_PATH)
print(f"Модель успешно сохранена в: {MODEL_PATH}")

Модель успешно сохранена в: steam_logistic_classification_model


Не забываем освобождать кешированные датафреймы и завершать `Spark`-сессию.

In [30]:
train_df.unpersist()
test_df.unpersist()
predictions.unpersist()
spark.stop()